In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader , Dataset , random_split
import re

In [2]:
with open(r"/content/1661-0.txt","r",encoding='utf-8') as file:
    text = file.read()


vocab = {"<UNK>":0}
idx2word = {}
content = text.lower()
content = re.sub(r"\s+"," " ,content)
content = content.strip()
content = content.split()
for txt in content:
    if txt not in vocab:
        vocab[txt] = len(vocab)
for key , value in vocab.items():
    idx2word[value] = key
indices = []
for txt in content:
    indices.append(vocab[txt])

X = []
y = []
seq_len =20
for i in range(len(indices)-seq_len):
    for j in range(i , i+seq_len):
        curr_indices = indices[i:j+1]
        padded_indices = [0] * (seq_len - len(curr_indices)) + curr_indices
        X.append(padded_indices)
        y.append(indices[j+1])


X = torch.tensor(X)
y = torch.tensor(y)

In [ ]:
print("Text: ",text[:1000])

Text:  ﻿
Project Gutenberg's The Adventures of Sherlock Holmes, by Arthur Conan Doyle

This eBook is for the use of anyone anywhere at no cost and with
almost no restrictions whatsoever.  You may copy it, give it away or
re-use it under the terms of the Project Gutenberg License included
with this eBook or online at www.gutenberg.net


Title: The Adventures of Sherlock Holmes

Author: Arthur Conan Doyle

Release Date: November 29, 2002 [EBook #1661]
Last Updated: May 20, 2019

Language: English

Character set encoding: UTF-8

*** START OF THIS PROJECT GUTENBERG EBOOK THE ADVENTURES OF SHERLOCK HOLMES ***



Produced by an anonymous Project Gutenberg volunteer and Jose Menendez



cover



The Adventures of Sherlock Holmes



by Arthur Conan Doyle



Contents


   I.     A Scandal in Bohemia
   II.    The Red-Headed League
   III.   A Case of Identity
   IV.    The Boscombe Valley Mystery
   V.     The Five Orange Pips
   VI.    The Man with the Twisted Lip
   VII.   The Adventure of th

In [ ]:
print("Tokens: ",content[:1000])

Tokens:  ['\ufeff', 'project', "gutenberg's", 'the', 'adventures', 'of', 'sherlock', 'holmes,', 'by', 'arthur', 'conan', 'doyle', 'this', 'ebook', 'is', 'for', 'the', 'use', 'of', 'anyone', 'anywhere', 'at', 'no', 'cost', 'and', 'with', 'almost', 'no', 'restrictions', 'whatsoever.', 'you', 'may', 'copy', 'it,', 'give', 'it', 'away', 'or', 're-use', 'it', 'under', 'the', 'terms', 'of', 'the', 'project', 'gutenberg', 'license', 'included', 'with', 'this', 'ebook', 'or', 'online', 'at', 'www.gutenberg.net', 'title:', 'the', 'adventures', 'of', 'sherlock', 'holmes', 'author:', 'arthur', 'conan', 'doyle', 'release', 'date:', 'november', '29,', '2002', '[ebook', '#1661]', 'last', 'updated:', 'may', '20,', '2019', 'language:', 'english', 'character', 'set', 'encoding:', 'utf-8', '***', 'start', 'of', 'this', 'project', 'gutenberg', 'ebook', 'the', 'adventures', 'of', 'sherlock', 'holmes', '***', 'produced', 'by', 'an', 'anonymous', 'project', 'gutenberg', 'volunteer', 'and', 'jose', 'menendez

In [ ]:
print("Vocab: ",vocab)
print("Index to word: ",idx2word)

Vocab:  {'<UNK>': 0, '\ufeff': 1, 'project': 2, "gutenberg's": 3, 'the': 4, 'adventures': 5, 'of': 6, 'sherlock': 7, 'holmes,': 8, 'by': 9, 'arthur': 10, 'conan': 11, 'doyle': 12, 'this': 13, 'ebook': 14, 'is': 15, 'for': 16, 'use': 17, 'anyone': 18, 'anywhere': 19, 'at': 20, 'no': 21, 'cost': 22, 'and': 23, 'with': 24, 'almost': 25, 'restrictions': 26, 'whatsoever.': 27, 'you': 28, 'may': 29, 'copy': 30, 'it,': 31, 'give': 32, 'it': 33, 'away': 34, 'or': 35, 're-use': 36, 'under': 37, 'terms': 38, 'gutenberg': 39, 'license': 40, 'included': 41, 'online': 42, 'www.gutenberg.net': 43, 'title:': 44, 'holmes': 45, 'author:': 46, 'release': 47, 'date:': 48, 'november': 49, '29,': 50, '2002': 51, '[ebook': 52, '#1661]': 53, 'last': 54, 'updated:': 55, '20,': 56, '2019': 57, 'language:': 58, 'english': 59, 'character': 60, 'set': 61, 'encoding:': 62, 'utf-8': 63, '***': 64, 'start': 65, 'produced': 66, 'an': 67, 'anonymous': 68, 'volunteer': 69, 'jose': 70, 'menendez': 71, 'cover': 72, 'cont

# Dataset creation

In [3]:
class CustomDataSet(Dataset):
    def __init__(self,X,y):
        self.X = X
        self.y = y
    def __len__(self):
        return self.X.shape[0]
    def __getitem__(self, idx):
        return self.X[idx] , self.y[idx]

data = CustomDataSet(X,y)
train_size = int(len(data) * 0.8)
splited_size = len(data) - train_size
train_data , splited_data = random_split(data , [train_size , splited_size])
test_size = int(len(splited_data) * 0.5)
val_size = len(splited_data) - test_size
test_data , val_data = random_split(splited_data , [test_size , val_size])
print(len(train_data)  , len(test_data) , len(val_data))

1721328 215166 215166


In [4]:
train_loader = DataLoader(train_data , batch_size=1024,shuffle=True)
test_loader = DataLoader(test_data , batch_size=1024, shuffle=False)
val_loader = DataLoader(val_data ,batch_size=1024,shuffle=False)

In [5]:
x , y = next(iter(train_loader))
criterion = nn.CrossEntropyLoss(ignore_index=0)
em = nn.Embedding(len(vocab),128)
multihead = nn.MultiheadAttention(embed_dim=128,num_heads=8,batch_first=True)
x = em(x)
contextual_em , softmax_weights = multihead(x,x,x)
# print(contextual_em.shape , x.shape)
lstm = nn.LSTM(input_size=128,hidden_size=256,num_layers=3,batch_first=True)
outputs , (h,c) = lstm(contextual_em)
fc = nn.Linear(256,len(vocab))
dense = fc(outputs[:,-1,:])

In [6]:
dense.shape

torch.Size([1024, 14557])

In [7]:
loss = criterion(dense , y)
loss.backward()
print(loss)

tensor(9.5851, grad_fn=<NllLossBackward0>)


In [14]:
class NextWordPredictor(nn.Module):
    def __init__(self, vocab , embeding_dim , num_layers, hidden_dim , number_head):
        super().__init__()
        self.embedding = nn.Embedding(len(vocab), embeding_dim)
        self.multi_head = nn.MultiheadAttention(embed_dim=embeding_dim ,
                                                num_heads=number_head,
                                                batch_first=True)
        self.lstm = nn.LSTM(input_size=embeding_dim , num_layers=num_layers,
                            hidden_size=hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim,len(vocab))
    def forward(self,x):
        x = self.embedding(x)
        contextual_em, softmax_weights = self.multi_head(x,x,x)
        enhanced_em = x + contextual_em # for semantic + contextual em , better training (prevent vanishing gradient)
        outputs , (h,c) = self.lstm(enhanced_em)
        out = self.fc(outputs[:,-1,:])
        return out

In [15]:
from tqdm import tqdm
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model= NextWordPredictor(vocab , 128, 3 , 256,2).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(),lr=0.001)
scheduler = optim.lr_scheduler.StepLR(optimizer,step_size=10,gamma=0.1)

number_epoch = 50
for epoch in range(number_epoch):
    train_loss = 0
    train_bar = tqdm(train_loader, desc=f"Train {epoch+1}/{number_epoch}",leave=False)
    for x , y in train_bar:
        x , y = x.to(device) , y.to(device)
        y_pred = model(x)
        loss = criterion(y_pred,y)
        optimizer.zero_grad()
        loss.backward()
        train_loss+=loss.item()
        optimizer.step()
        train_bar.set_postfix(loss=loss.item())
    train_loss/= len(train_loader)
    scheduler.step()
    print(f"{epoch+1}/{number_epoch} : Train loss {train_loss}")

1/50 : Train loss 6.398049406747177


2/50 : Train loss 4.92219876732449


3/50 : Train loss 3.9876654749466933


4/50 : Train loss 3.2953921628097635


5/50 : Train loss 2.7835962219226933


...

46/50 : Train loss 0.9840383603060834


47/50 : Train loss 0.9840251348473925


48/50 : Train loss 0.9840127365494115


49/50 : Train loss 0.9840002669456954


50/50 : Train loss 0.9839870202477528


In [16]:
def calculate_accuracy(y_pred , y):
  y_pred = y_pred.argmax(dim=1)
  corrected = (y_pred == y).sum()
  acc = corrected / y.shape[0]
  return acc

test_loss = 0
test_accuracy = 0
test_bar = tqdm(test_loader, desc=f"Test",leave=False)
for x , y in test_bar:
    x , y = x.to(device) , y.to(device)
    y_pred = model(x)
    loss = criterion(y_pred,y)
    optimizer.zero_grad()
    loss.backward()
    test_loss+=loss.item()
    test_accuracy += calculate_accuracy(y_pred , y)
    train_bar.set_postfix(loss=loss.item())
test_loss/= len(test_loader)
test_accuracy /=len(test_loader)
print(f"\nTest loss {test_loss}")
print(f"\nTest accuracy {test_accuracy}")


Test loss 1.2474990953201366

Test accuracy 0.7736177444458008


In [17]:
model.eval()
with torch.no_grad():
    for X_batch, y_batch in test_loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)
        outputs = model(X_batch)
        preds = torch.argmax(outputs, dim=1)
        X_batch = X_batch.cpu()
        y_batch = y_batch.cpu()
        preds = preds.cpu()
        for i in range(len(X_batch)):
            sentence = [idx2word[idx.item()] for idx in X_batch[i] if idx.item() in idx2word]
            predicted_word = idx2word[preds[i].item()]
            actual_word = idx2word[y_batch[i].item()]
            print("Input Sentence:", " ".join(sentence))
            print("Predicted Word:", predicted_word)
            print("Actual Word:", actual_word)
            print("-"*50)

Streaming output truncated to the last 5000 lines.
Input Sentence: <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> give your
Predicted Word: reasons,”
Actual Word: reasons,”
--------------------------------------------------
Input Sentence: me to leave the room while you explain this matter?” “if i may give an opinion,” remarked the strange gentleman,
Predicted Word: “we’ve
Actual Word: “we’ve
--------------------------------------------------
Input Sentence: <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> “we
Predicted Word: have
Actual Word: were
--------------------------------------------------
Input Sentence: <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> in animated conversation with two men, one of whom
Predicted Word: i
Actual Word: i
--------------------------------------------------
Input Sentence: <UNK> <UNK> <UNK> <UNK> <UNK> <UNK>

In [18]:
import random
def generate_next_words(model, sequence, idx2word, seq_len=20, predict_n=20, device="cpu"):
    model.eval()
    sequence = [idx for idx in sequence if idx != 0]
    generated_words = []
    with torch.no_grad():
        for _ in range(predict_n):
            input_seq = sequence[-seq_len:]
            if len(input_seq) < seq_len:
                input_seq = [0]*(seq_len-len(input_seq)) + input_seq
            x = torch.tensor(input_seq).unsqueeze(0).to(device)
            output = model(x)
            pred_idx = torch.argmax(output, dim=1).item()
            sequence.append(pred_idx)
            generated_words.append(idx2word.get(pred_idx, "<UNK>"))
    return generated_words

sentence_number = 50
for X_batch, y_batch in test_loader:
    for i in range(sentence_number):
      idx = random.randint(0,len(X_batch))
      seq = X_batch[idx].tolist()
      real_sentence = [idx2word[i] for i in seq if i != 0]
      predicted_words = generate_next_words(
          model,seq,idx2word,seq_len=20,predict_n=20,device=device
      )
      print("Input sentence:")
      print(" ".join(real_sentence))
      print("\nGenerated next 20 words:")
      print(" ".join(predicted_words))
      print("--"*10)

    break

Input sentence:
early that morning

Generated next 20 words:
a peasant had met a cart containing several people and some new face, “but certainly not a very quick-witted youth,
--------------------
Input sentence:
are

Generated next 20 words:
the very word,” said holmes. “it is very clear.” “if you remember, monday was quite a very good man, too.
--------------------
Input sentence:
of interest. the only drawback is that there is no

Generated next 20 words:
law, i fear, that we are not accustomed to give you the chance.” “my dear fellow, i congratulate you.” “i
--------------------
Input sentence:
expression upon his aristocratic features. “my messenger reached you, then?” asked holmes. “yes, and i confess that the contents

Generated next 20 words:
startled me beyond measure. have you nothing in them.” “but my mistress told me that you were engaged.” “so we
--------------------
Input sentence:
have been some attempt to hide the discoloured

Generated next 20 words:
patches by smeari